In [1]:

import pandas as pd
import numpy as np



In [2]:
df = pd.read_csv("/home/patel-krish/anaconda3/BrainyBeam/heart.csv")



In [3]:


columns = ["age", "trestbps", "chol", "thalach", "oldpeak"]

# Function to count outliers using IQR
def count_outliers(data):
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    return ((data < lower) | (data > upper)).sum()



In [4]:
original = df[columns].copy()



In [5]:
winsorized = original.copy()

for col in columns:
    lower = original[col].quantile(0.05)
    upper = original[col].quantile(0.95)

    winsorized[col] = original[col].clip(
        lower=lower,
        upper=upper
    )



In [6]:

log_transformed = original.copy()

for col in columns:
    log_transformed[col] = np.log1p(original[col])



In [7]:
comparison = pd.DataFrame({
    "Feature": columns,
    "Outliers Before": [
        count_outliers(original[col]) for col in columns
    ],
    "Outliers After Winsorization": [
        count_outliers(winsorized[col]) for col in columns
    ],
    "Skewness Before": [
        original[col].skew() for col in columns
    ],
    "Skewness After Winsorization": [
        winsorized[col].skew() for col in columns
    ],
    "Skewness After Log": [
        log_transformed[col].skew() for col in columns
    ]
})

print(comparison.round(3))



    Feature  Outliers Before  Outliers After Winsorization  Skewness Before  \
0       age                0                             0           -0.249   
1  trestbps               30                             0            0.740   
2      chol               16                             0            1.074   
3   thalach                4                             0           -0.514   
4   oldpeak                7                             0            1.211   

   Skewness After Winsorization  Skewness After Log  
0                        -0.237              -0.625  
1                         0.383               0.305  
2                         0.288               0.082  
3                        -0.380              -0.980  
4                         0.798               0.358  


In [9]:
winsorized.to_csv("heart_winsorized.csv", index=False)
log_transformed.to_csv("heart_log_transformed.csv", index=False)
